In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from datetime import date
import collections
import datetime
import os
import xarray as xr
from cycler import cycler
import matplotlib.patches as mpatches

# Path to the CSV file in Google Drive (streaming)
file_path = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/Lizz Research Stuff/City Scale Analysis/South America/SouthAmerica.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

In [ ]:
def get_glacier_ids(city_name, input_df):
    """
    Retrieve glacier IDs for a given city from the DataFrame.
    
    Parameters:
    - city_name (str): The name of the city (e.g. 'La Paz' or 'Santiago').
    - input_df (DataFrame): The DataFrame containing glacier IDs.

    Returns:
    - List of glacier IDs for the specified city.
    """
    # Ensure the city name is valid
    if city_name not in input_df.columns:
        raise ValueError("City name not found in DataFrame columns")

    # Get the column for the specified city
    column = input_df[city_name]

    # Drop NaN values and return the list of glacier IDs
    glacier_ids = column.dropna().astype(str).apply(lambda x: x[-8:]).tolist()

    return glacier_ids

In [ ]:
LaPaz_IDs = get_glacier_ids('La Paz', df)
Santiago_IDs = get_glacier_ids('Santiago', df)
Quito_IDs = get_glacier_ids('Quito', df)

Reading in GloGEM Projections for South America:

In [ ]:
# Base paths for the regions
fpath16 = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/Lizz Research Stuff/Runoff-intercomparison/GloGEM-output/RGI16-LowLatitudes/files/'
fpath17 = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/Lizz Research Stuff/Runoff-intercomparison/GloGEM-output/RGI17-SouthernAndes/files/'

#All of the climate models used
modelnames = ['BCC-CSM2-MR', 'CESM2', 'CESM2-WACCM', 'EC-Earth3', 'EC-Earth3-Veg', 'FGOALS-f3-L', 'GFDL-ESM4', 
                  'INM-CM4-8', 'INM-CM5-0', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorESM2-MM']

scenarios = ['ssp126','ssp245','ssp370','ssp585']   #Specifiying the SSP
#SSPs = ['ssp119','ssp126','ssp245','ssp370','ssp585'] #Use a different path as we have all 5 ssps for volume

In [ ]:
# Initialize nested lists for discharges
all_discharges = [[] for _ in scenarios]

for s, SSP in enumerate(scenarios):
    model_discharges = []
    for modelname in modelnames:
        # Read the discharge data for Region 16 (Low Latitudes)
        temp_df16 = pd.read_csv(fpath16 + modelname + '/' + scenarios[s] + '/' + 'lowlatitudes_Discharge_r1.dat', 
                               sep='\s+', header=None, skiprows=1, index_col=0)
        temp_df16.index = temp_df16.index.map(lambda x: str(x).zfill(5))
        temp_df16.index = '16.' + temp_df16.index.astype(str)

        # Read the discharge data for Region 17 (Southern Andes)
        temp_df17 = pd.read_csv(fpath17 + modelname + '/' + scenarios[s] + '/' + 'southernandes_Discharge_r1.dat', 
                                sep='\s+', header=None, skiprows=1, index_col=0)
        temp_df17.index = temp_df17.index.map(lambda x: str(x).zfill(5))
        temp_df17.index = '17.' + temp_df17.index.astype(str)

        # Combine the data for both regions
        temp_df = pd.concat([temp_df16, temp_df17])

        # Filter to only include rows with glacier IDs in LaPaz_IDs or Santiago_IDs
        temp_df = temp_df.loc[temp_df.index.isin(LaPaz_IDs + Santiago_IDs)]

        # Append the filtered DataFrame to the list for this model
        model_discharges.append(temp_df)
    
    # Store the filtered model discharges for the current SSP path
    all_discharges[s] = model_discharges

In [ ]:
# Create new index using pandas date_range function
start_date = datetime.date(1980, 1, 1)
end_date = datetime.date(2100, 12, 1)
new_indices = pd.date_range(start_date, end_date, freq='MS').strftime('%Y-%m').tolist()

# Apply new index and datetime conversion
for s, SSP_discharges in enumerate(all_discharges):
    for m, discharge_df in enumerate(SSP_discharges):
        all_discharges[s][m].columns = new_indices
        all_discharges[s][m].columns = pd.to_datetime(new_indices)

In [ ]:
#GloGEM does not have Quito glaciers (run to check)
#all_discharges[s][m].loc['16.01339']
#all_discharges[s][m].loc['16.02943']

In [ ]:
runoff = {s: {m: None for m in modelnames} for s in scenarios}  # create nested dictionary indexed by model name and ssp
all_areas = {s: {m: None for m in modelnames} for s in scenarios}

for s, SSP in enumerate(scenarios):
    for m, modelname in enumerate(modelnames):
        # Read the area data for Region 16
        temp_df16 = pd.read_csv(fpath16 + modelname + '/' + scenarios[s] + '/' + 'lowlatitudes_Area_r1.dat', 
                               sep='\s+', index_col="ID")
        temp_df16.index = temp_df16.index.map(lambda x: str(x).zfill(5))
        temp_df16.index = '16.' + temp_df16.index.astype(str)

        # Read the area data for Region 17 (Southern Andes)
        temp_df17 = pd.read_csv(fpath17 + modelname + '/' + scenarios[s] + '/' + 'southernandes_Area_r1.dat', 
                                sep='\s+', index_col="ID")
        temp_df17.index = temp_df17.index.map(lambda x: str(x).zfill(5))
        temp_df17.index = '17.' + temp_df17.index.astype(str)

        # Combine the data for both regions
        temp_df = pd.concat([temp_df16, temp_df17])

        temp_df = temp_df.loc[temp_df.index.isin(LaPaz_IDs + Santiago_IDs)]

        all_areas[SSP][modelname] = temp_df

        new_df = all_areas[SSP][modelname].iloc[:, 0].values.repeat(all_discharges[s][m].shape[1]).reshape(all_discharges[s][m].shape)
        initial_areas = pd.DataFrame(new_df, index=all_discharges[s][m].index, columns=all_discharges[s][m].columns).mul(1e6)
        runoff[SSP][modelname] = pd.concat([initial_areas * all_discharges[s][m]], axis=1) * 1e-9  #m^3 to km^3

In [ ]:
city_dict = {
    'La Paz': LaPaz_IDs,
    'Santiago': Santiago_IDs,
    'Quito': Quito_IDs
    # Add more cities here as needed
}

In [ ]:
def get_city_runoff(city_name, SSP, modelname):
    """
    Calculate the total runoff for a given city.
    
    Parameters:
    - city_name (str): The name of the city (e.g., 'La Paz' or 'Santiago').
    - SSP (str): The SSP scenario (e.g., 'ssp126').
    - modelname (str): The climate model name (e.g., 'CESM2').
    
    Returns:
    - Pandas Series with aggregated runoff for the specified city in km^3.
    """
    # Ensure the city name is valid
    if city_name not in city_dict:
        raise ValueError("City name not found in city_dict list")
    
    # Get the list of glacier IDs for the specified city
    city_ids = city_dict[city_name]

    # Retrieve the runoff DataFrame for the specified SSP and model
    runoff_df = runoff[SSP][modelname]

    # Filter the runoff DataFrame to include only the rows with glacier IDs in city_ids
    city_runoff_df = runoff_df.loc[runoff_df.index.isin(city_ids)]

    # Aggregate the runoff values by summing across all glaciers in the city's list
    aggregated_runoff = city_runoff_df.sum()

    return aggregated_runoff

In [ ]:
# Initialize a dictionary to store the aggregated runoff projections for each city
city_runoff_glo = {city: {s: {m: None for m in modelnames} for s in scenarios} for city in city_dict}

# Loop through each city, SSP, and model to calculate and store the aggregated runoff projections
for city_name in city_dict.keys():
    for SSP in scenarios:
        for modelname in modelnames:
            # Calculate the aggregated runoff for the current city, SSP, and model
            aggregated_runoff = get_city_runoff(city_name, SSP, modelname)
            
            # Store the aggregated runoff in the dictionary
            city_runoff_glo[city_name][SSP][modelname] = aggregated_runoff

Reading in OGGM Projections for South America:

In [ ]:
Alpine_basins = {'TITICACA':'3912', 'SANTA':'3425', 'OCONA':'3418', 'MAJES':'3416', 
                'MAGDALENA':'3227', 'AMAZON':'3203','YELCHO':'3429', 'VALDIVIA':'3428', 'SERRANO':'3426', 'RAPEL':'3423', 
                 'PUELO':'3422','PASCUA':'3420', 'PALENA':'3419', 'HUASCO':'3412', 'COPIAPO':'3409', 'CISNES':'3408', 
                'BIOBIO':'3405', 'BAKER':'3404', 'AZOPARDO':'3403', 'AISEN':'3401', 'SANTA CRUZ':'3244', 
                'NEGRO':'3232', 'COLORADO':'3212', 'CHICO':'3209'}

basins = ['TITICACA', 'SANTA', 'OCONA', 'MAJES', 'MAGDALENA', 'AMAZON', 'YELCHO', 'VALDIVIA', 'SERRANO','RAPEL','PUELO', 'PASCUA', 'PALENA', 'HUASCO', 'COPIAPO', 
          'CISNES', 'BIOBIO', 'BAKER', 'AZOPARDO', 'AISEN', 'SANTA CRUZ', 'NEGRO', 'COLORADO', 'CHICO']

SSPs = ['ssp126','ssp245','ssp370','ssp585']

#Generic filepath to navigate to Drive folder 
fpathOG1 = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/'
fpathOG2 = 'Lizz Research Stuff/Runoff-intercomparison/OGGM/lschuster/runs_2023.3/output/basins/'

In [ ]:
# Combine all city IDs into a single list for filtering
all_city_ids = []
for city_name, city_ids in city_dict.items():
    all_city_ids.extend(city_ids)

# Function to filter glacier IDs during loading
def filter_glacier_ids(ds, city_ids):
    filtered_ds = ds.sel(rgi_id=[ID for ID in ds.rgi_id.values if ID[-8:] in city_ids])
    return filtered_ds

In [ ]:
# Initialize dictionary for storing filtered runoff data
rf_ds_filtered = {}
rf_ds_monthly_filtered = {}

# Importing all runoff data, OGGM is grouped by basin
for basin, ID in Alpine_basins.items():
    fpath_basin = 'gcm_from_2000_bc_2000_2019/{}/'.format(ID)
    
    # Load and filter the annual runoff data
    with xr.open_mfdataset(f'{fpathOG1 + fpathOG2 + fpath_basin}*.nc') as ds1:
        ds1 = ds1.runoff.load()
        ds1_filtered = filter_glacier_ids(ds1, all_city_ids)
        rf_ds_filtered[basin] = ds1_filtered
    
    # Load and filter the monthly runoff data
    with xr.open_mfdataset(f'{fpathOG1 + fpathOG2 + fpath_basin}*.nc') as ds2:
        ds_monthly = ds2.runoff_monthly.load()
        ds_monthly_filtered = filter_glacier_ids(ds_monthly, all_city_ids)
        rf_ds_monthly_filtered[basin] = ds_monthly_filtered

In [ ]:
# Initialize the final dictionary to store the aggregated city runoff data
city_runoff_OG = {city_name: {SSP: {modelname: None for modelname in modelnames} for SSP in SSPs} for city_name in city_dict}

# Loop over SSPs and model names
for SSP in SSPs:
    for modelname in modelnames:
        # Initialize a temporary dictionary to hold the filtered DataArrays by basin
        filtered_dataarrays = []
        
        # Loop over each basin in the filtered data
        for basin, ds in rf_ds_monthly_filtered.items():
            # Check if the DataArray has relevant rgi_id's
            if ds.sel(time=slice('2000-01-01', '2019-12-31')).notnull().any():
                filtered_dataarrays.append(ds)
        
        # Combine the filtered DataArrays into one DataArray
        if filtered_dataarrays:
            combined_da = xr.concat(filtered_dataarrays, dim="rgi_id")
            
            # Loop over each city and aggregate the runoff
            for city_name, city_ids in city_dict.items():
                city_ids_filtered = [ID for ID in combined_da.rgi_id.values if ID[-8:] in city_ids]
                
                if city_ids_filtered:
                    city_da = combined_da.sel(rgi_id=city_ids_filtered)
                    city_runoff_OG[city_name][SSP] = city_da.sum(dim="rgi_id") * 1e-12 #convert from kg to km^3
                else:
                    city_runoff_OG[city_name][SSP] = xr.DataArray()

Reading in PyGEM Projections for South America:

In [ ]:
fpathPy = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/Lizz Research Stuff/Runoff-intercomparison/PyGEM/'

In [ ]:
#Importing all runoff data, taking annual sum, and converting m^3 to km^3
import glob   #use glob to group files by filename similarities (in this case, SSP)

regions = ['16', '17']

rf_ds = {}
for s, SSP in enumerate(SSPs):
    rf_ds[SSP] = {}
    for region in regions:
        fpath1 = f"{region}/R{region}_runoff_monthly_c2_ba1_1set_2000_2100-{SSP}"
        file_pattern = f'{fpathPy + fpath1}*.nc'
        file_list = glob.glob(file_pattern)
        #print(file_list)
        
        datasets = []  # Create an empty list for each SSP
        if file_list:
            for file in file_list:
                with xr.open_dataset(file) as ds:
                    ds = ds.glac_runoff_monthly.load()
                    datasets.append(ds)
    
            combined_ds = xr.concat(datasets, dim='glacier')  # Concatenate the datasets
            rf_ds[SSP][region] = combined_ds

# Concatenate datasets along the 'glacier' dimension (ensure this dimension exists)
rf_ds_combined = {}
for SSP in SSPs:
    combined_ds = xr.concat([rf_ds[SSP]['16'], rf_ds[SSP]['17']], dim='glacier')
    rf_ds_combined[SSP] = combined_ds

In [ ]:
def filter_glacier_ids(ds, city_ids):
    # Extract the last eight characters of each RGIId in the dataset
    rgi_ids_short = ds.RGIId.str.slice(-8)
    filtered_ds = ds.where(rgi_ids_short.isin(city_ids), drop=True)
    #Return matching values
    return filtered_ds

In [ ]:
# Applying the filter
city_runoff_Py = {}
for city_name, city_ids in city_dict.items(): 
    city_runoff_Py[city_name] = {}
    for SSP in SSPs:
        temp_rf = filter_glacier_ids(rf_ds_combined[SSP], city_ids)
        city_runoff_Py[city_name][SSP] = temp_rf.sum(dim="glacier") * 1e-9 #convert from m^3 to km^3

Making data formats match across all glacier models:

In [ ]:
gmodels = ['GloGEM', 'OGGM', 'PyGEM']

cities = ['La Paz', 'Quito', 'Santiago']

date_range = pd.date_range(start='2000-01-01', periods=1212, freq='M')

# Initialize the nested dictionary using comprehension
all_rf_data = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}
all_rf_data_annual = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}

# Populate the dictionary with the data
for gmodel in gmodels:
    for city_name in cities: 
        for SSP in scenarios:
            m=0
            for m, GCM in enumerate(modelnames):
                if city_name != 'Santiago' and gmodel == 'OGGM':
                    temp_series = pd.Series(city_runoff_OG[city_name][SSP].sel(gcm=GCM, scenario=SSP).values.flatten(), index=date_range)
                    all_rf_data[gmodel][city_name][SSP][GCM] = temp_series
                    all_rf_data_annual[gmodel][city_name][SSP][GCM] = all_rf_data[gmodel][city_name][SSP][GCM].resample('A').sum()
                elif gmodel == 'GloGEM':
                    all_rf_data['GloGEM'][city_name][SSP][GCM] = city_runoff_glo[city_name][SSP][GCM][240::]
                    all_rf_data['GloGEM'][city_name][SSP][GCM].index = date_range
                    all_rf_data_annual[gmodel][city_name][SSP][GCM] = all_rf_data['GloGEM'][city_name][SSP][GCM].resample('A').sum()
                elif gmodel == 'PyGEM':
                    temp_series = pd.Series(city_runoff_Py[city_name][SSP].sel(model=m + 1).values.flatten(), index=date_range)
                    all_rf_data[gmodel][city_name][SSP][GCM] = temp_series
                    all_rf_data_annual[gmodel][city_name][SSP][GCM] = all_rf_data[gmodel][city_name][SSP][GCM].resample('A').sum()

In [ ]:
city_runoff_glo['Quito'][SSP][GCM][240::]

Calculating multi-GCM medians and inter-quartile ranges:

In [ ]:
GCM_mean = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}
GCM_q1 = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}
GCM_q3 = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}

# Taking multi-GCM means and quartiles
for g, gmodel in enumerate(gmodels):
    for city_name in cities: 
        for s, SSP in enumerate(scenarios):
            if gmodel == 'OGGM' and city_name == 'Santiago':
                continue
            else:
                df_dict = all_rf_data_annual[gmodel][city_name][SSP]
                df_mean = pd.concat(df_dict.values(), axis=1).mean(axis=1)
                GCM_mean[gmodel][city_name][SSP] = df_mean
    
                df_q1 = pd.concat(df_dict.values(), axis=1).quantile(q=0.25, axis=1)
                GCM_q1[gmodel][city_name][SSP] = df_q1
    
                df_q3 = pd.concat(df_dict.values(), axis=1).quantile(q=0.75, axis=1)
                GCM_q3[gmodel][city_name][SSP] = df_q3

Single City Plot

In [ ]:
#Creating time values and color schemes
yrs = np.arange(2000, 2101)
yrs_dt = pd.to_datetime([str(y) for y in yrs])

colorschemes = {}

colors_glo =  plt.colormaps['Greens']
line_colors_glo = colors_glo(np.linspace(0.2, 0.6, num = 12))
glo_cycler = cycler(color = line_colors_glo)
colorschemes['GloGEM'] = glo_cycler

colors_OG =  plt.colormaps['Blues']
line_colors_OG = colors_OG(np.linspace(0.2, 0.6,num = 12))
OG_cycler = cycler(color = line_colors_OG)
colorschemes['OGGM'] = OG_cycler

colors_py =  plt.colormaps['Purples']
line_colors_py = colors_py(np.linspace(0.2, 0.6,num = 12))
py_cycler = cycler(color = line_colors_py)
colorschemes['PyGEM'] = py_cycler

colors = {'GloGEM': 'darkgreen', 'OGGM': 'royalblue', 'PyGEM': 'purple'}
fill_colors = {'GloGEM': 'green', 'OGGM': 'dodgerblue', 'PyGEM': 'purple'}

In [ ]:
# Input a single city name
city = 'La Paz'

# Create a 1-row, 4-column subplot arrangement
fig, axs = plt.subplots(1, 4, figsize=(13, 2.5), sharex=True)

for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels):
        for m, GCM in enumerate(modelnames):

            axs[s].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][city][SSP][GCM][0:-1], color=axs[s].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[s].plot(yrs_dt[0:-1], GCM_mean[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[s].plot(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[s].plot(yrs_dt[0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[s].fill_between(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=fill_colors[gmodel])
            axs[s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))

            # Setting x and y labels
            #axs[s].set_xlabel('Year')
            axs[s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation = 45)

            axs[s].set_xlabel('Year')
            #axs[s].set_xticks([pd.to_datetime('2025-01-01'), pd.to_datetime('2050-01-01'), pd.to_datetime('2075-01-01'), pd.to_datetime('2100-01-01')])
            #axs[s].set_xticks([pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')])


        # Setting y limits uniform within basins
        if s == 0:
            axs[s].set_ylabel(city + r' $[km^3]$')
        else:
            axs[s].set_ylabel(None)
            axs[s].set_yticklabels('')

        # Add the SSP title
        axs[s].set_title(SSP, fontsize=10)

# Limits determined by max/min between all GCMs
row_min = np.inf
row_max = -np.inf
for s in range(len(scenarios)):
    data_min = np.min(axs[s].get_ybound()[0])
    data_max = np.max(axs[s].get_ybound()[1])
    if data_min < row_min:
        row_min = data_min
    if data_max > row_max:
        row_max = data_max

for s in range(len(scenarios)):
    axs[s].set_ylim(row_min, row_max)

# Legend
green_patch = mpatches.Patch(color='darkgreen', label='GloGEM')
purple_patch = mpatches.Patch(color='purple', label='PyGEM')
blue_patch = mpatches.Patch(color='royalblue', label='OGGM')
#axs[0].legend(handles=[green_patch, purple_patch, blue_patch], bbox_to_anchor=(3.15, 1.2), ncol=3)

# Titles
#plt.suptitle(f'{basin} Runoff Projections for Major Southern Andes River Basins', x=0.5, y=0.9)
#plt.title('SSP 126                            SSP 245                           SSP 370                             SSP 585', x=-1.28, y=1)
# Show the plot
name = city
#plt.savefig(f"/Users/finnwimberly/Desktop/Lizz Research/AGU Figures/{name}.png", dpi=300, bbox_inches='tight')


plt.show()

In [ ]:
# Input a single city name
city = 'Quito'

# Create a 1-row, 4-column subplot arrangement
fig, axs = plt.subplots(1, 4, figsize=(13, 2.5), sharex=True)

for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels):
        for m, GCM in enumerate(modelnames):
            axs[s].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][city][SSP][GCM][0:-1], color=axs[s].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[s].plot(yrs_dt[0:-1], GCM_mean[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[s].plot(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[s].plot(yrs_dt[0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[s].fill_between(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=fill_colors[gmodel])
            axs[s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))

            # Setting x and y labels
            #axs[s].set_xlabel('Year')
            axs[s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation = 45)

            axs[s].set_xlabel('Year')
            #axs[s].set_xticks([pd.to_datetime('2025-01-01'), pd.to_datetime('2050-01-01'), pd.to_datetime('2075-01-01'), pd.to_datetime('2100-01-01')])
            #axs[s].set_xticks([pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')])


        # Setting y limits uniform within basins
        if s == 0:
            axs[s].set_ylabel(city + r' $[km^3]$')
        else:
            axs[s].set_ylabel(None)
            axs[s].set_yticklabels('')

        # Add the SSP title
        axs[s].set_title(SSP, fontsize=10)

# Limits determined by max/min between all GCMs
row_min = np.inf
row_max = -np.inf
for s in range(len(scenarios)):
    data_min = np.min(axs[s].get_ybound()[0])
    data_max = np.max(axs[s].get_ybound()[1])
    if data_min < row_min:
        row_min = data_min
    if data_max > row_max:
        row_max = data_max

for s in range(len(scenarios)):
    axs[s].set_ylim(row_min, row_max)

# Legend
green_patch = mpatches.Patch(color='darkgreen', label='GloGEM')
purple_patch = mpatches.Patch(color='purple', label='PyGEM')
blue_patch = mpatches.Patch(color='royalblue', label='OGGM')
#axs[0].legend(handles=[green_patch, purple_patch, blue_patch], bbox_to_anchor=(3.15, 1.2), ncol=3)

# Titles
#plt.suptitle(f'{basin} Runoff Projections for Major Southern Andes River Basins', x=0.5, y=0.9)
#plt.title('SSP 126                            SSP 245                           SSP 370                             SSP 585', x=-1.28, y=1)
# Show the plot
name = city
#plt.savefig(f"/Users/finnwimberly/Desktop/Lizz Research/AGU Figures/{name}.png", dpi=300, bbox_inches='tight')


plt.show()

No projections available for the two glaciers within the Quito watershed

In [ ]:
# Input a single city name
city = 'Santiago'
gmodels_noOG = ['GloGEM', 'PyGEM']

# Create a 1-row, 4-column subplot arrangement
fig, axs = plt.subplots(1, 4, figsize=(13, 2.5), sharex=True)

for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels_noOG):
        for m, GCM in enumerate(modelnames):
            axs[s].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][city][SSP][GCM][0:-1], color=axs[s].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[s].plot(yrs_dt[0:-1], GCM_mean[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[s].plot(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[s].plot(yrs_dt[0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[s].fill_between(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=fill_colors[gmodel])
            axs[s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))

            # Setting x and y labels
            #axs[s].set_xlabel('Year')
            axs[s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation = 45)

            axs[s].set_xlabel('Year')
            #axs[s].set_xticks([pd.to_datetime('2025-01-01'), pd.to_datetime('2050-01-01'), pd.to_datetime('2075-01-01'), pd.to_datetime('2100-01-01')])
            #axs[s].set_xticks([pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')])


        # Setting y limits uniform within basins
        if s == 0:
            axs[s].set_ylabel(city + r' $[km^3]$')
        else:
            axs[s].set_ylabel(None)
            axs[s].set_yticklabels('')

        

# Limits determined by max/min between all GCMs
row_min = np.inf
row_max = -np.inf
for s in range(len(scenarios)):
    data_min = np.min(axs[s].get_ybound()[0])
    data_max = np.max(axs[s].get_ybound()[1])
    if data_min < row_min:
        row_min = data_min
    if data_max > row_max:
        row_max = data_max

for s in range(len(scenarios)):
    axs[s].set_ylim(row_min, row_max)

# Legend
green_patch = mpatches.Patch(color='darkgreen', label='GloGEM')
purple_patch = mpatches.Patch(color='purple', label='PyGEM')
blue_patch = mpatches.Patch(color='royalblue', label='OGGM')
#axs[0].legend(handles=[green_patch, purple_patch, blue_patch], bbox_to_anchor=(3.15, 1.2), ncol=3)

# Titles
#plt.suptitle(f'{basin} Runoff Projections for Major Southern Andes River Basins', x=0.5, y=0.9)
#plt.title('SSP 126                            SSP 245                           SSP 370                             SSP 585', x=-1.28, y=1)
# Show the plot
name = city
#plt.savefig(f"/Users/finnwimberly/Desktop/Lizz Research/AGU Figures/{name}.png", dpi=300, bbox_inches='tight')


plt.show()

Creating RF figures of % change:

In [ ]:
#Generating data
percent_change = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}

for gmodel in gmodels:
    for city in cities:
        for SSP in scenarios:
            if city == 'Santiago' and gmodel == 'OGGM':
                continue
            else:    
                percent_change[gmodel][city][SSP] = ((GCM_mean[gmodel][city][SSP]- GCM_mean[gmodel][city][SSP][0:20].mean()) /  GCM_mean[gmodel][city][SSP][0:20].mean())*100

In [ ]:
# Input a single city name
city = 'La Paz'

# Create a 1-row, 4-column subplot arrangement
fig, axs = plt.subplots(1, 4, figsize=(13, 2.5), sharex=True)

for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels):
        axs[s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
        axs[s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1], color=colors[gmodel], alpha=0.3, linewidth=0.7)
        
    axs[s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))

    # Setting x and y labels
    #axs[s].set_xlabel('Year')
    axs[s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation = 45)

    axs[s].set_xlabel('Year')
    #axs[s].set_xticks([pd.to_datetime('2025-01-01'), pd.to_datetime('2050-01-01'), pd.to_datetime('2075-01-01'), pd.to_datetime('2100-01-01')])
    #axs[s].set_xticks([pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')])


    # Setting y limits uniform within basins
    if s == 0:
        axs[s].set_ylabel(r' Percent Change [%]')
    else:
        axs[s].set_ylabel(None)
        axs[s].set_yticklabels('')

    # Add the SSP title
    axs[s].set_title(SSP, fontsize=10)

In [ ]:
# Input a single city name
city = 'Quito'

# Create a 1-row, 4-column subplot arrangement
fig, axs = plt.subplots(1, 4, figsize=(13, 2.5), sharex=True)

for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels):
        axs[s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
        axs[s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1], color=colors[gmodel], alpha=0.3, linewidth=0.7)
        
    axs[s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))

    # Setting x and y labels
    #axs[s].set_xlabel('Year')
    axs[s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation = 45)

    axs[s].set_xlabel('Year')
    #axs[s].set_xticks([pd.to_datetime('2025-01-01'), pd.to_datetime('2050-01-01'), pd.to_datetime('2075-01-01'), pd.to_datetime('2100-01-01')])
    #axs[s].set_xticks([pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')])


    # Setting y limits uniform within basins
    if s == 0:
        axs[s].set_ylabel(r' Percent Change [%]')
    else:
        axs[s].set_ylabel(None)
        axs[s].set_yticklabels('')

    # Add the SSP title
    axs[s].set_title(SSP, fontsize=10)

In [ ]:
# Input a single city name
city = 'Santiago'

# Create a 1-row, 4-column subplot arrangement
fig, axs = plt.subplots(1, 4, figsize=(13, 2.5), sharex=True)

for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels):
        if gmodel == 'OGGM':
            continue
        else:
            axs[s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
            axs[s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1], color=colors[gmodel], alpha=0.3, linewidth=0.7)
        
    axs[s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))

    # Setting x and y labels
    #axs[s].set_xlabel('Year')
    axs[s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation = 45)

    axs[s].set_xlabel('Year')
    #axs[s].set_xticks([pd.to_datetime('2025-01-01'), pd.to_datetime('2050-01-01'), pd.to_datetime('2075-01-01'), pd.to_datetime('2100-01-01')])
    #axs[s].set_xticks([pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')])


    # Setting y limits uniform within basins
    if s == 0:
        axs[s].set_ylabel(r' Percent Change [%]')
    else:
        axs[s].set_ylabel(None)
        axs[s].set_yticklabels('')

    # Add the SSP title
    axs[s].set_title(SSP, fontsize=10)

Lets make composite plots for each city:

In [ ]:
city = 'Santiago'
gmodels_noOG = ['GloGEM', 'PyGEM']

# Create a 2-row, 4-column subplot arrangement
fig, axs = plt.subplots(2, 4, figsize=(13, 5), sharex=True)

# Top row: First plot
for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels_noOG):
        for m, GCM in enumerate(modelnames):
            axs[0, s].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][city][SSP][GCM][0:-1], color=axs[0, s].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[0, s].plot(yrs_dt[0:-1], GCM_mean[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[0, s].plot(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[0, s].plot(yrs_dt[0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[0, s].fill_between(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=fill_colors[gmodel])
            axs[0, s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[0, s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            if s == 0:
                axs[0, s].set_ylabel(r'Annual Runoff $[km^3]$')
            axs[0, s].set_title(SSP, fontsize=10)

# Bottom row: Second plot
for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels_noOG):
        axs[1, s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
        axs[1, s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1], color=colors[gmodel], alpha=0.3, linewidth=0.7)
        axs[1, s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
        axs[1, s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
        axs[1, s].set_xlabel('Year')
        if s == 0:
            axs[1, s].set_ylabel(r'Percent Change [%]')
        axs[1, s].set_title(SSP, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
city = 'La Paz'

# Create a 2-row, 4-column subplot arrangement
fig, axs = plt.subplots(2, 4, figsize=(13, 5), sharex=True)

# Top row: First plot
for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels):
        for m, GCM in enumerate(modelnames):
            axs[0, s].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][city][SSP][GCM][0:-1], color=axs[0, s].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[0, s].plot(yrs_dt[0:-1], GCM_mean[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[0, s].plot(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[0, s].plot(yrs_dt[0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[0, s].fill_between(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=fill_colors[gmodel])
            axs[0, s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[0, s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            if s == 0:
                axs[0, s].set_ylabel(r'Annual Runoff $[km^3]$')
            axs[0, s].set_title(SSP, fontsize=10)

# Bottom row: Second plot
for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels):
        axs[1, s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
        axs[1, s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1], color=colors[gmodel], alpha=0.3, linewidth=0.7)
        axs[1, s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
        axs[1, s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
        axs[1, s].set_xlabel('Year')
        if s == 0:
            axs[1, s].set_ylabel(r'Percent Change [%]')
        axs[1, s].set_title(SSP, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
city = 'Quito'
gmodels_noGlo = ['OGGM', 'PyGEM']

# Create a 2-row, 4-column subplot arrangement
fig, axs = plt.subplots(2, 4, figsize=(13, 5), sharex=True)

# Top row: First plot
for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels_noGlo):
        for m, GCM in enumerate(modelnames):
            axs[0, s].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][city][SSP][GCM][0:-1], color=axs[0, s].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[0, s].plot(yrs_dt[0:-1], GCM_mean[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[0, s].plot(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[0, s].plot(yrs_dt[0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[0, s].fill_between(yrs_dt[0:-1], GCM_q1[gmodel][city][SSP][0:-1], GCM_q3[gmodel][city][SSP][0:-1], color=fill_colors[gmodel])
            axs[0, s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[0, s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            if s == 0:
                axs[0, s].set_ylabel(r'Annual Runoff $[km^3]$')
            axs[0, s].set_title(SSP, fontsize=10)

# Bottom row: Second plot
for s, SSP in enumerate(scenarios):
    for g, gmodel in enumerate(gmodels_noGlo):
        axs[1, s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
        axs[1, s].plot(yrs_dt[0:-1], percent_change[gmodel][city][SSP][0:-1], color=colors[gmodel], alpha=0.3, linewidth=0.7)
        axs[1, s].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
        axs[1, s].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
        axs[1, s].set_xlabel('Year')
        if s == 0:
            axs[1, s].set_ylabel(r'Percent Change [%]')
        axs[1, s].set_title(SSP, fontsize=10)

plt.tight_layout()
plt.show()

Writing out data:

In [ ]:
output_dir =  '/Users/finnwimberly/Desktop/Subbasin Examination/Processed Data/'

# Loop through the scenarios, basins, models, and GCMs
for SSP in scenarios:
    for gmodel in gmodels:
        for GCM in modelnames:
            for city in cities:
                if gmodel == 'OGGM' and city == 'Santiago':
                    continue
                else:
                    # Create a list to store the data for this combination
                    data_list = []
                    
                    # Collect the data
                    for i, year in enumerate(yrs_dt[0:-1]):
                        annual_runoff = all_rf_data_annual[gmodel][city][SSP][GCM][i]
                        percent_change_val = percent_change[gmodel][city][SSP][i]
                        data_list.append({
                            'Year': year,
                            'Annual Runoff [km^3]': annual_runoff,
                            'Percent Change [%]': percent_change_val
                        })
        
                    # Convert the data list to a DataFrame
                    df = pd.DataFrame(data_list)
        
                    # Define the file name based on the combination
                    fname = f"Subbasin_RF_{GCM}_{SSP}_{gmodel}_{city}.csv"
        
                    # Define the full path of the output file
                    output_path = os.path.join(output_dir, fname)
        
                    # Save the DataFrame as CSV
                    df.to_csv(output_path, header=True, index=False)